# Large-window snATAC-seq integration

ATAC data use iterative LSI rather than the RNA linear autoencoder, followed by the same public `ot.integrate()` API. The prior expiring signed URL has been replaced by stable Figshare downloads.

- ATAC AnnData: [large_atac_windows.h5ad](https://ndownloader.figshare.com/files/25721780)
- Gene annotation: [GENCODE mouse vM25](https://ndownloader.figshare.com/files/59759393)
- Processing background: [Muon chromatin-accessibility tutorial](https://muon-tutorials.readthedocs.io/en/latest/single-cell-rna-atac/pbmc10k/2-Chromatin-Accessibility-Processing.html)


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
path = fetch("large_atac_windows.h5ad", "https://ndownloader.figshare.com/files/25721780")
adata = subsample(sc.read_h5ad(path), 20_000)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
batch_key = "batchname_all" if "batchname_all" in adata.obs else "batch"
adata


## Iterative LSI and OT


In [ ]:
scb.pp.find_variable_features(adata, batch_key=batch_key)
scb.pp.add_iterative_lsi(
    adata, n_components=31, n_iter=2, topN=min(50_000, adata.n_vars),
    per_cluster_union=False, drop_first_component=True, add_key="X_lsi",
)
adata, metrics = scb.ot.integrate(
    adata, obsm_key="X_lsi", batch_key=batch_key, out_key="X_ot",
    modality="atac", random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


In [ ]:
sc.pp.neighbors(adata, use_rep="X_ot", metric="cosine", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
colors = [batch_key] + (["final_cell_label"] if "final_cell_label" in adata.obs else [])
sc.pl.umap(adata, color=colors)
